In [2]:
# module imports
from glob import glob
import pandas as pd
import os
import numpy as np
from scipy.io import loadmat
import scipy.sparse as sp

In [ ]:
# set dirs
data_dir = "/mnt/labworlds/Hayden/Hayden_Lab/speech_247/podcast_comparison"

In [4]:
# get patient dirs
patient_dirs = glob(f"{data_dir}/Y*")
patient_dirs = {os.path.basename(dir_):dir_ for dir_ in patient_dirs}

In [5]:
# load spikes and transcripts
patient_spikes = {}
for pt, dir_ in patient_dirs.items():
    spike_path = glob(f"{dir_}/spikes/*binary_spiketrain.npy")[0]
    spiketrain = np.load(spike_path)
    patient_spikes[pt] = spiketrain

In [ ]:
# load transcripts
transcript_dfs = {}
for pt, dir_ in patient_dirs.items():
    transcript_path = glob(f"{dir_}/whisperx_out/audio_transcript_df.csv")[0]
    transcript_df = pd.read_csv(transcript_path, index_col=0)
    transcript_dfs[pt] = transcript_df
    

In [7]:
import numpy as np
from scipy.signal import fftconvolve

def gaussian_firing_rate(
    spike_bin,          # (n_channels, n_bins) uint8/float; 1s where spikes occur
    fs=1000,            # Hz; your spike_bin rate
    sigma_ms=50.0,      # Gaussian std in milliseconds (typical: 10–100 ms)
    truncate=3.0,       # kernel half-width in sigmas (3→ ~99.7% mass)
    edge_mode="reflect" # 'reflect', 'constant', 'nearest'; for padding
):
    """
    Return: rates_hz with shape (n_channels, n_bins), smoothed firing rate in Hz.
    """
    spike_bin = np.asarray(spike_bin, dtype=float)
    n_ch, n_bins = spike_bin.shape

    # build Gaussian kernel in bins
    sigma_bins = sigma_ms * fs / 1000.0
    radius = int(np.ceil(truncate * sigma_bins))
    t = np.arange(-radius, radius + 1, dtype=float)
    kernel = np.exp(-0.5 * (t / sigma_bins) ** 2)
    kernel /= kernel.sum()  # area = 1 (in bins)

    # pad for edge handling
    def pad1d(x):
        if edge_mode == "reflect":
            left = x[:, 1:radius+1][:, ::-1]
            right = x[:, -radius-1:-1][:, ::-1]
        elif edge_mode == "nearest":
            left = np.repeat(x[:, :1], radius, axis=1)
            right = np.repeat(x[:, -1:], radius, axis=1)
        elif edge_mode == "constant":
            left = np.zeros((x.shape[0], radius))
            right = np.zeros((x.shape[0], radius))
        else:
            raise ValueError("edge_mode must be 'reflect', 'nearest', or 'constant'")
        return np.concatenate([left, x, right], axis=1)

    xpad = pad1d(spike_bin)

    # FFT convolution per channel
    rates = np.empty_like(spike_bin, dtype=float)
    for c in range(n_ch):
        y = fftconvolve(xpad[c], kernel, mode="same")
        # unpad back to original length
        rates[c] = y[:,][radius:-radius]

    # convert from spikes/bin to spikes/s
    rates_hz = rates * fs
    return rates_hz

In [8]:
# now get spike counts/frs from word onset to offset
patient_frs = {}
patient_counts = {}
patient_durs = {}
for pt, df in transcript_dfs.items():
    word_frs = []
    word_counts = []
    word_durs = []
    spikes = patient_spikes[pt]
    gaussian_frs = gaussian_firing_rate(spikes)
    
    for idx, row in df.iterrows():
        # now get onset/offset index of word
        word_start = np.round(row.start_ts * 1000).astype(int)
        word_end = np.round(row.end_ts * 1000).astype(int)

        # now get spike counts and frs
        word_spikes = spikes[:, word_start:word_end].sum(axis=1)
        frs = gaussian_frs[:, word_start:word_end].mean(axis=1)

        word_counts.append(word_spikes)
        word_frs.append(frs)
        word_durs.append(word_end - word_start)

    word_frs = np.vstack(word_frs)
    word_counts = np.vstack(word_counts)
    word_durs = np.array(word_durs)
    patient_frs[pt] = word_frs
    patient_counts[pt] = word_counts
    patient_durs[pt] = word_durs

In [9]:
# now save all our stuff
for pt, spikes in patient_frs.items():
    np.save(f"{data_dir}/{pt}/whisperx_out/whisperx_spikes.npy", spikes)
    frs = patient_frs[pt]
    np.save(f"{data_dir}/{pt}/whisperx_out/whisperx_frs.npy", frs)
    durs = patient_durs[pt]
    np.save(f"{data_dir}/{pt}/whisperx_out/whisperx_durs.npy", durs)

In [10]:
# get w2v embeddings
model_dir = "/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_out"
import fasttext
import fasttext.util


ft = fasttext.load_model(f'{model_dir}/cc.en.300.bin')

In [11]:
# now get w2v embeddings for all these words
# now add semantic categories to patient transcripts
import string
w2v_embeddings = {}
for pt, df in transcript_dfs.items():
    pt_embeddings = []
    for idx, row in df.iterrows():
        try:
            word = row.word.lower().strip(string.punctuation).strip()
            embedding = ft.get_word_vector(word)
            # try to output embedding, otherwise just nan vector
        except Exception as e:
            print(e)
            embedding = np.full_like(embedding, np.nan)
        pt_embeddings.append(embedding)
    pt_embeddings = np.vstack(pt_embeddings)
    w2v_embeddings[pt] = pt_embeddings


In [12]:
# also get semantic categories
# now load semantic category classifier
import dill as pickle
with open(f"{model_dir}/semantic_cat_classifier.pkl", "rb") as f:
    model = pickle.load(f)

/scratch/tahaismail424/miniforge3/envs/speech_247_env/lib/python3.11/site-packages/joblib/_multiprocessing_helpers.py:44: UserWarning: [Errno 13] Permission denied.  joblib will operate in serial mode
  warnings.warn("%s.  joblib will operate in serial mode" % (e,))
/scratch/tahaismail424/miniforge3/envs/speech_247_env/lib/python3.11/site-packages/dill/_dill.py:452: UserWarning: [16:07:20] WARNING: /workspace/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  obj = StockUnpickler.load(self)


In [13]:
# now add semantic category prediction for each patient
threshold = 0.8
for pt, embeddings in w2v_embeddings.items():
    transcript_df = transcript_dfs[pt]
    data_probs = model.predict_proba(embeddings)
    top_idx = data_probs.argmax(axis=1)
    conf = data_probs[np.arange(data_probs.shape[0]), top_idx]
    keep = conf >= threshold
    top_idx[~keep] = -1
    unique_semantic_cats = top_idx
    transcript_df['semantic_category_pred'] = unique_semantic_cats
    transcript_dfs[pt] = transcript_df

In [14]:
# resave
for pt, dir_ in patient_dirs.items():
    transcript_df = transcript_dfs[pt]
    transcript_df.to_csv(f'{dir_}/whisperx_out/audio_transcript_df.csv')

In [15]:
# also get gpt embeddings
# login to hugginface
from huggingface_hub import login

# Replace "hf_YOUR_ACCESS_TOKEN" with your actual token
login(token=os.environ['HF_TOKEN'])

/scratch/tahaismail424/miniforge3/envs/speech_247_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
# device
import torch
device = torch.device("cuda")

In [17]:
# set huggingface home and load gemma model
from transformers import GPT2Tokenizer, GPT2Model
import torch

os.environ["HF_HOME"] = "/scratch/tahaismail424/hf"
tokenizer = GPT2Tokenizer.from_pretrained('gpt2-large')
model = GPT2Model.from_pretrained('gpt2-large').to(device).eval()


In [18]:
def rfind_sublist(lst, sub):
    """Return the starting index of the last occurrence of `sub` in `lst`, or -1 if not found."""
    n, m = len(lst), len(sub)
    for i in range(n - m, -1, -1):  # start from last possible match
        if lst[i:i + m] == sub:
            return i
    return -1

In [19]:
# potentially add option to remove punctuation while searching
def get_average_embedding(embeddings, token_list, word):
    # starting from end of token list, do linear search for the world we're looking for
    # tokenize our particular word
    enc = tokenizer(word, return_tensors='pt', add_special_tokens=True)
    tok_rep = tokenizer.convert_ids_to_tokens(enc.input_ids[0])
    # remove [CLS] [SEP] tokens
    tok_rep = tok_rep
    idx = rfind_sublist(token_list, tok_rep)
    word_embedding = np.nanmean(embeddings[:, idx:idx+len(tok_rep), :], axis=1)
    return word_embedding
    

In [20]:
from tqdm import tqdm
gpt_embeddings = {}
for pt, transcript_df in transcript_dfs.items():
    embeddings = []
    # now get embeddings for each word
    model.eval()
    for idx, row in tqdm(transcript_df.iterrows(), total=len(transcript_df)):
        try:
            text = row.context + ' ' + row.word if (isinstance(row.context, str) and row.context != '') else row.word
            enc = tokenizer(text, return_tensors='pt', add_special_tokens=True).to(device)
            # try to output embedding, otherwise just nan vector
            with torch.no_grad():
                model_out = model(**enc, output_hidden_states=True)
                hidden_states = np.array([layer.squeeze().cpu().numpy() for layer in model_out[2]])
                if len(hidden_states.shape) == 2:
                    embedding = hidden_states
                else:
                    # get tokens
                    all_toks = tokenizer.convert_ids_to_tokens(enc.input_ids[0])
                    to_embed = row.word if row.context == '' else ' ' + row.word
                    embedding = get_average_embedding(hidden_states, all_toks, to_embed)
        except Exception as e:
            print(e)
            embedding = np.full_like(embedding, np.nan)
        embeddings.append(embedding)
    embeddings = np.array(embeddings)
    gpt_embeddings[pt] = embeddings

  0%|          | 0/6942 [00:00<?, ?it/s]/tmp/ipykernel_727567/569214936.py:10: RuntimeWarning: Mean of empty slice
  word_embedding = np.nanmean(embeddings[:, idx:idx+len(tok_rep), :], axis=1)
100%|██████████| 6979/6979 [03:03<00:00, 38.09it/s]


In [21]:
# now save embeddings
for pt, embeddings in gpt_embeddings.items():
    np.save(f"{data_dir}/{pt}/whisperx_out/whisperx_gpt_embeddings.npy", embeddings)